In [3]:
import requests

url = 'https://arxiv.org/pdf/2210.03629.pdf'   # URL файла
filename = 'react.pdf'        # Желаемое имя файла

response = requests.get(url, stream=True)  # stream=True для больших файлов
response.raise_for_status()                # Проверка на ошибки

with open(filename, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

print(f'Файл сохранён как {filename}')

Файл сохранён как react.pdf


In [4]:
import os
from langchain_community.document_loaders.pdf import PDFPlumberLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PDFPlumberLoader(filename)
data = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=150)
chunks = text_splitter.split_documents(data)

d:\Projects\RKSP_Kursach\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
chunks

[Document(metadata={'source': 'react.pdf', 'file_path': 'react.pdf', 'page': 0, 'total_pages': 33, 'Author': '', 'CreationDate': 'D:20230313000911Z', 'Creator': 'LaTeX with hyperref', 'Keywords': '', 'ModDate': 'D:20230313000911Z', 'PTEX.Fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'Producer': 'pdfTeX-1.40.21', 'Subject': '', 'Title': '', 'Trapped': 'False'}, page_content='PublishedasaconferencepaperatICLR2023\nREACT: SYNERGIZING REASONING AND ACTING IN\nLANGUAGE MODELS\nShunyuYao∗*,1,JeffreyZhao2,DianYu2,NanDu2,IzhakShafran2,KarthikNarasimhan1,YuanCao2\n1DepartmentofComputerScience,PrincetonUniversity\n2GoogleResearch,Brainteam\n1{shunyuy,karthikn}@princeton.edu\n2{jeffreyzhao,dianyu,dunan,izhak,yuancao}@google.com\nABSTRACT\nWhilelargelanguagemodels(LLMs)havedemonstratedimpressiveperformance\nacross tasks in language understanding and interactive decision making, their\nabilitiesforreasoning(e.g. chain-of-thoughtprompting)andac

In [6]:
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings

from qdrant_client import QdrantClient, models
from qdrant_client.http.models import Distance, SparseVectorParams, VectorParams
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode

embeddings = FastEmbedEmbeddings()
bm25_model = FastEmbedSparse(model_name="Qdrant/BM25")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]d:\Projects\RKSP_Kursach\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\furmi\AppData\Local\Temp\fastembed_cache\models--qdrant--bge-small-en-v1.5-onnx-q. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fetching 18 files:   0%|          | 0/18

In [7]:
collection_name = "research-app"
client = QdrantClient(url="http://localhost:6333")

In [9]:
client.create_collection(
    collection_name=collection_name,
    vectors_config={"Dense": VectorParams(size=384, distance=Distance.COSINE,on_disk=True)},
    sparse_vectors_config={"Sparse": SparseVectorParams(index=models.SparseIndexParams(on_disk=False))}
)

True

In [11]:
vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings,
    sparse_embedding=bm25_model,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="Dense",
    sparse_vector_name="Sparse",
)

In [12]:
vector_store.add_documents(documents=chunks)


retriever = vector_store.as_retriever(search_kwargs={"k":4})

In [13]:
SYSTEM_TEMPLATE = """
You are an expert QA Assistant who answers questions using only the provided context as your source of information.
If the question is not from the provided context, say `I don't know. Not enough information received.`
"""

HUMAN_TEMPLATE = """
We have provided context information below.

CONTEXT: {context_str}
---------------------
Given this information, please answer the question: {query}
---------------------
If the question is not from the provided context, say `I don't know. Not enough information received.`
"""

In [25]:
from langchain_gigachat.chat_models import GigaChat
from langgraph.graph import StateGraph, START, END

from langchain_core.documents import Document
from typing_extensions import List, TypedDict

class RAGState(TypedDict):
    question: str
    context: List[Document]
    answer: str

In [39]:
from dotenv import load_dotenv
load_dotenv()
credentials = os.environ.get('GIGACHAT_CREDENTIALS')

llm = GigaChat(
    credentials=credentials,
    verify_ssl_certs=False,
)

In [40]:
def search(state: RAGState):
  retrieved_docs = vector_store.max_marginal_relevance_search(state["question"])
  return {"context": retrieved_docs}

def generate(state: RAGState):
  docs_content = "\n\n".join(doc.page_content for doc in state["context"])
  messages = [
      {"role": "system", "content": SYSTEM_TEMPLATE},
      {"role": "user", "content": HUMAN_TEMPLATE.format(context_str=docs_content, query=state["question"])},
  ]
  response = llm.invoke(messages)
  return {"answer": response.content}

In [33]:
graph_node = StateGraph(RAGState).add_sequence([search, generate])
graph_node.add_edge(START, "search")
graph_node.add_edge("search", "generate")
graph_node.add_edge("generate", END)
graph = graph_node.compile()

In [34]:
response1 = graph.invoke({"question": "how to evaluate React agent?"})
print(response1['answer'])
response2 = graph.invoke({"question": "What is the meaning of life?"})
print(response2['answer'])

To evaluate the ReAct agent, we can consider several aspects based on the provided context:

1. **Performance Improvements**: The initial promising results indicate that learning from more high-quality human annotations will be crucial for improving its performance.
   
2. **Multi-task Training**: Scaling up ReAct through multi-task training could enhance its capabilities.

3. **Combining Paradigms**: Combining ReAct with complementary paradigms like reinforcement learning may lead to stronger agents capable of unlocking the full potential of large language models (LLMs).

4. **Error Analysis**: Understanding error cases such as "reasoning errors" where the model fails to reason about proper next actions and jumps out of loops can help identify areas for improvement.

5. **Retrieval Quality**: Successful retrieval of informative knowledge via search is critical; non-informative searches account for a significant portion of errors (23%).

6. **Prompting Methods**: The best prompting met